# Notebook 0 — YOLOv11m (fine-tuned) vs Gemma 3 27B-IT (zero-shot)
### Deteccao binaria de residuos em vias publicas brasileiras

Pipeline reprodutivel de ponta a ponta: preparacao de dados, treino do detector
especializado, inferencia do VLM generalista, avaliacao comparativa (metricas,
significancia, latencia) e analise qualitativa.

**Reprodutibilidade.** Seeds fixas em `random`, `numpy`, `torch`/`cudnn`, no treino
do YOLO (`seed=42`, `deterministic=True`) e no VLM (`temperature=0`, `seed=42`
quando suportado). Hardware e versoes sao registrados em `outputs/metricas_finais.json`.

**Escopo.** Comparacao de **estrategias de implantacao** (detector ajustado a dados
publicos x VLM generalista sem ajuste), nao de superioridade arquitetural.

## 1. Configuracao e reprodutibilidade

In [ ]:
# Instalacao (descomente na primeira execucao)
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
# !pip install ultralytics>=8.3 roboflow>=1.1 openai>=1.30 python-dotenv>=1.0
# !pip install opencv-python>=4.10 Pillow>=10.0 pillow-heif>=0.16 imagehash>=4.3
# !pip install jsonschema>=4.0 numpy>=1.26 pandas>=2.2 scikit-learn>=1.4
# !pip install matplotlib>=3.8 seaborn>=0.13 tqdm>=4.66 PyYAML>=6.0 statsmodels>=0.14

In [ ]:
import os
# Definido antes dos demais imports para registrar a intencao de determinismo.
os.environ.setdefault("PYTHONHASHSEED", "42")

import sys, json, time, shutil, hashlib, base64, random, subprocess, warnings, io
import importlib.metadata as importlib_metadata
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import yaml
import cv2
from PIL import Image, ImageOps
import pillow_heif
pillow_heif.register_heif_opener()

from dotenv import load_dotenv
load_dotenv()

warnings.filterwarnings("ignore")

# --- Determinismo global ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

EXTS_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic"}
EXTS_MANIFESTO_DADOS = EXTS_IMG | {".txt", ".yaml", ".csv"}

def listar_imagens(pasta) -> list:
    pasta = Path(pasta)
    if not pasta.exists():
        return []
    return sorted(p for p in pasta.rglob("*")
                  if p.is_file() and p.suffix.lower() in EXTS_IMG)

def contar_imagens(pasta) -> int:
    return len(listar_imagens(pasta))

def limpar_diretorio(pasta):
    pasta = Path(pasta)
    if pasta.exists():
        shutil.rmtree(pasta)
    pasta.mkdir(parents=True, exist_ok=True)

def pacote_versao(distribuicao: str):
    try:
        return importlib_metadata.version(distribuicao)
    except importlib_metadata.PackageNotFoundError:
        return None

def versoes_pacotes():
    pacotes = {
        "numpy": "numpy", "pandas": "pandas", "torch": "torch",
        "ultralytics": "ultralytics", "roboflow": "roboflow",
        "openai": "openai", "opencv_python": "opencv-python",
        "Pillow": "Pillow", "pillow_heif": "pillow-heif",
        "imagehash": "ImageHash", "jsonschema": "jsonschema",
        "scikit_learn": "scikit-learn", "statsmodels": "statsmodels",
        "matplotlib": "matplotlib", "seaborn": "seaborn", "PyYAML": "PyYAML",
    }
    return {nome: pacote_versao(dist) for nome, dist in pacotes.items()}

def git_commit(repo_dir):
    repo_dir = Path(repo_dir)
    if not (repo_dir / ".git").exists():
        return None
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
    except Exception:
        return None

def sha256_arquivo(path, bloco=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bloco), b""):
            h.update(chunk)
    return h.hexdigest()

def hash_manifesto_diretorio(root, extensoes=None):
    root = Path(root)
    if not root.exists():
        return {"existe": False, "n_arquivos": 0, "sha256_manifesto": None}
    linhas = []
    for p in sorted(x for x in root.rglob("*") if x.is_file()):
        if extensoes is not None and p.suffix.lower() not in extensoes:
            continue
        linhas.append(f"{p.relative_to(root).as_posix()}\t{p.stat().st_size}\t{sha256_arquivo(p)}")
    payload = "\n".join(linhas).encode("utf-8")
    return {"existe": True, "n_arquivos": len(linhas),
            "sha256_manifesto": hashlib.sha256(payload).hexdigest()}

def ler_imagem_bgr(path):
    # PIL cobre HEIC/WebP/PNG; o array BGR preserva a convencao usada pelo OpenCV/Ultralytics.
    with Image.open(path) as im:
        rgb = np.array(ImageOps.exif_transpose(im).convert("RGB"))
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

def imagem_para_data_url_jpeg(path, quality=95):
    with Image.open(path) as im:
        rgb = ImageOps.exif_transpose(im).convert("RGB")
        buf = io.BytesIO()
        rgb.save(buf, format="JPEG", quality=quality)
    payload = buf.getvalue()
    b64 = base64.b64encode(payload).decode()
    return {
        "data_url": f"data:image/jpeg;base64,{b64}",
        "mime_type": "image/jpeg",
        "payload_bytes": len(payload),
        "payload_sha256": hashlib.sha256(payload).hexdigest(),
    }

print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | "
      f"Pandas {pd.__version__} | Torch {torch.__version__}")

In [ ]:
# Dispositivo
assert torch.cuda.is_available(), "CUDA nao disponivel. Verifique a instalacao do PyTorch."
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU_NAME} | VRAM: {VRAM_GB:.1f} GB | CUDA: {torch.version.cuda}")

In [ ]:
# Variaveis de ambiente (.env) e diretorios
VARS_OBRIGATORIAS = [
    "ROBOFLOW_API_KEY",
    "ROBOFLOW_WORKSPACE_TRASH",    "ROBOFLOW_PROJECT_TRASH",    "ROBOFLOW_VERSION_TRASH",
    "ROBOFLOW_WORKSPACE_SIDEWALK", "ROBOFLOW_PROJECT_SIDEWALK", "ROBOFLOW_VERSION_SIDEWALK",
    "ROBOFLOW_WORKSPACE_TESTE",    "ROBOFLOW_PROJECT_TESTE",    "ROBOFLOW_VERSION_TESTE",
    "LM_STUDIO_BASE_URL", "LM_STUDIO_MODEL", "YOLO_PRETRAINED",
]
_ausentes = [v for v in VARS_OBRIGATORIAS if not os.getenv(v)]
if _ausentes:
    raise EnvironmentError(f"Variaveis ausentes no .env: {_ausentes}")

for d in [
    "data/raw_taco", "data/raw_trash_detection", "data/raw_sidewalk", "data/raw_imagens_teste",
    "data/unified/images/train", "data/unified/images/val",
    "data/unified/labels/train", "data/unified/labels/val",
    "data/teste/images", "data/teste/labels",
    "models", "outputs/vlm_cache", "outputs/figures",
]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("Ambiente e diretorios verificados.")

## 2. Download dos datasets

- **TACO**: repositorio oficial (GitHub); anotacoes COCO, convertidas na Secao 3.
- **Trash Detection**: Roboflow, formato YOLO nativo.
- **Sidewalk Segmentation**: Roboflow; apenas imagens (background, mascaras descartadas).
- **imagens-teste**: Roboflow; ~560 imagens reais de ruas brasileiras, exclusivas para teste.

In [ ]:
# TACO: git clone + script oficial de download
TACO_REPO = Path("data/raw_taco/TACO")
if not TACO_REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/pedropro/TACO.git", str(TACO_REPO)], check=True)
print("Baixando imagens TACO (links Flickr mortos sao ignorados)...")
subprocess.run([sys.executable, "download.py"], check=False, cwd=str(TACO_REPO))
print(f"TACO: {contar_imagens(TACO_REPO)} imagens disponiveis.")

In [ ]:
# Roboflow: Trash Detection, Sidewalk (background) e conjunto de teste
from roboflow import Roboflow
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))

def baixar_roboflow(ws_var, proj_var, ver_var, destino):
    ws, proj, ver = os.getenv(ws_var), os.getenv(proj_var), int(os.getenv(ver_var))
    rf.workspace(ws).project(proj).version(ver).download("yolov8", location=destino, overwrite=True)
    return contar_imagens(destino)

DATASETS_RF = [
    ("ROBOFLOW_WORKSPACE_TRASH",    "ROBOFLOW_PROJECT_TRASH",    "ROBOFLOW_VERSION_TRASH",    "data/raw_trash_detection"),
    ("ROBOFLOW_WORKSPACE_SIDEWALK", "ROBOFLOW_PROJECT_SIDEWALK", "ROBOFLOW_VERSION_SIDEWALK", "data/raw_sidewalk"),
    ("ROBOFLOW_WORKSPACE_TESTE",    "ROBOFLOW_PROJECT_TESTE",    "ROBOFLOW_VERSION_TESTE",    "data/raw_imagens_teste"),
]
for ws_var, proj_var, ver_var, destino in DATASETS_RF:
    n = baixar_roboflow(ws_var, proj_var, ver_var, destino)
    print(f"{destino}: {n} imagens")

## 3. Deduplicacao e unificacao das classes

Deduplicacao perceptual (pHash, distancia <= 5 bits) no pool de treino. Em seguida,
unificacao dos datasets positivos na classe unica `lixo` (id 0), com background do
Sidewalk (`.txt` vazios) e **split agrupado 85/15** (augmentations do mesmo original
nunca cruzam treino/val).

In [ ]:
import imagehash

def detectar_duplicatas_perceptuais(image_paths, threshold=5):
    # Retorna pares (path_a, path_b, distancia) com pHash a distancia <= threshold.
    # Caminho completo (str) como chave: evita colisao entre datasets com arquivos homonimos.
    hashes, duplicatas = {}, []
    for path in tqdm(image_paths, desc="pHash", leave=False):
        try:
            h = imagehash.phash(Image.open(path).convert("RGB"))
        except Exception:
            continue
        for h_exist, p_exist in hashes.items():
            if h - h_exist <= threshold:
                duplicatas.append((str(p_exist), str(path), h - h_exist))
        hashes[h] = path
    return duplicatas

def coletar_imagens(pastas):
    imgs = []
    for pasta in pastas:
        imgs += listar_imagens(pasta)
    return sorted(set(imgs))

In [ ]:
# Deduplicacao do pool de treino
imgs_treino = coletar_imagens(["data/raw_taco", "data/raw_trash_detection", "data/raw_sidewalk"])
print(f"Pool de treino bruto: {len(imgs_treino)} imagens")
dups_treino = detectar_duplicatas_perceptuais(imgs_treino)
REMOVER_TREINO = {b for _, b, _ in dups_treino}
print(f"Duplicatas no treino: {len(dups_treino)} pares | imagens a remover: {len(REMOVER_TREINO)}")

In [ ]:
# Deduplicacao do conjunto de teste (apenas reportada — remocao manual se necessario)
dups_teste = detectar_duplicatas_perceptuais(coletar_imagens(["data/raw_imagens_teste"]))
print(f"Duplicatas no teste: {len(dups_teste)} pares "
      f"({'nenhuma' if not dups_teste else 'verificar manualmente'}).")

In [ ]:
# Conversao TACO COCO -> YOLO (classe unica)
def converter_taco_coco_para_yolo(taco_dir: Path) -> int:
    candidatos = list(taco_dir.rglob("annotations.json"))
    if not candidatos:
        raise FileNotFoundError(f"annotations.json nao encontrado em {taco_dir}")
    ann_path = candidatos[0]
    data_dir = ann_path.parent
    coco = json.load(open(ann_path))
    img_info = {img["id"]: img for img in coco["images"]}
    ann_por_img = defaultdict(list)
    for ann in coco["annotations"]:
        ann_por_img[ann["image_id"]].append(ann)

    n_labels = 0
    for img_id, anns in tqdm(ann_por_img.items(), desc="COCO->YOLO", leave=False):
        meta = img_info.get(img_id)
        if meta is None:
            continue
        img_path = data_dir / meta["file_name"]
        if not img_path.exists():
            continue
        W, H = meta.get("width") or 0, meta.get("height") or 0
        if W == 0 or H == 0:
            try:
                with Image.open(img_path) as im:
                    W, H = im.size
            except Exception:
                continue
        linhas = []
        for ann in anns:
            x_min, y_min, w, h = ann["bbox"]
            xc = max(0.0, min(1.0, (x_min + w / 2) / W))
            yc = max(0.0, min(1.0, (y_min + h / 2) / H))
            wn = max(0.0, min(1.0, w / W))
            hn = max(0.0, min(1.0, h / H))
            if wn > 0 and hn > 0:
                linhas.append(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
        img_path.with_suffix(".txt").write_text("\n".join(linhas))
        n_labels += 1
    return n_labels

n_taco_conv = converter_taco_coco_para_yolo(Path("data/raw_taco"))
print(f"Conversao TACO: {n_taco_conv} arquivos .txt criados.")

In [ ]:
# Funcoes de unificacao
CLASSES_OPERACIONAIS = {"lixo": 0}

def reescrever_label_lixo(lbl_path):
    # Reescreve qualquer classe original para a classe unica 0 (lixo).
    if not lbl_path or not lbl_path.exists():
        return []
    novas = []
    for linha in lbl_path.read_text().strip().split("\n"):
        parts = linha.strip().split()
        if len(parts) < 5:
            continue
        novas.append("0 " + " ".join(parts[1:]))
    return novas

def extrair_chave_grupo(nome_arquivo: str) -> str:
    # Chave de grupo para o split agrupado: agrupa augmentations do Roboflow
    # (mesmo original, hashes .rf. distintos) preservando o ID numerico unico.
    stem = Path(nome_arquivo).stem.split(".rf.")[0]
    for suf in ["_aug", "_flip", "_rot", "_mosaic", "_copy"]:
        stem = stem.split(suf)[0]
    return stem

def nome_unico(img_path: Path, prefixo: str) -> str:
    if prefixo == "taco_":
        return prefixo + "_".join(img_path.parts[-2:])   # inclui o batch do TACO
    return f"{prefixo}{img_path.name}"

In [ ]:
# Unificacao + split agrupado 85/15
UNIFIED_IMG_TRAIN = Path("data/unified/images/train")
UNIFIED_IMG_VAL   = Path("data/unified/images/val")
UNIFIED_LBL_TRAIN = Path("data/unified/labels/train")
UNIFIED_LBL_VAL   = Path("data/unified/labels/val")

for d in [UNIFIED_IMG_TRAIN, UNIFIED_IMG_VAL, UNIFIED_LBL_TRAIN, UNIFIED_LBL_VAL]:
    limpar_diretorio(d)

# 1) Coletar positivos (TACO + Trash), respeitando a deduplicacao
pares_positivos = []
for pasta, prefixo in [("data/raw_taco", "taco_"), ("data/raw_trash_detection", "trash_")]:
    p = Path(pasta)
    splits = [s for s in ["train", "valid", "test"] if (p / s / "images").exists()]
    if splits:
        for s in splits:
            for img_path in listar_imagens(p / s / "images"):
                if str(img_path) in REMOVER_TREINO:
                    continue
                lbl = p / s / "labels" / (img_path.stem + ".txt")
                pares_positivos.append((img_path, lbl if lbl.exists() else None, prefixo))
    else:
        for img_path in listar_imagens(p):
            if str(img_path) in REMOVER_TREINO:
                continue
            lbl = img_path.with_suffix(".txt")
            pares_positivos.append((img_path, lbl if lbl.exists() else None, prefixo))

# 2) Limitar TACO (amostragem semeada) para equilibrar as fontes
N_ALVO_TACO = int(os.getenv("TACO_N_ALVO", "1500"))
taco_pares   = [x for x in pares_positivos if x[2] == "taco_"]
outros_pares = [x for x in pares_positivos if x[2] != "taco_"]
if len(taco_pares) > N_ALVO_TACO:
    idx = np.random.default_rng(SEED).choice(len(taco_pares), size=N_ALVO_TACO, replace=False)
    taco_pares = [taco_pares[i] for i in sorted(idx)]
pares_positivos = taco_pares + outros_pares
print(f"Positivos: {len(pares_positivos)} (TACO={len(taco_pares)}, Trash={len(outros_pares)})")

# 3) Selecionar background (~15% do total) do Sidewalk
PROP_BG_ALVO = 0.15
n_bg_alvo = round(PROP_BG_ALVO / (1 - PROP_BG_ALVO) * len(pares_positivos))
sidewalk_bruto = listar_imagens("data/raw_sidewalk")
sidewalk_disp = [p for p in sidewalk_bruto if str(p) not in REMOVER_TREINO]
n_sidewalk_removidos = len(sidewalk_bruto) - len(sidewalk_disp)
if n_sidewalk_removidos:
    print(f"Background Sidewalk removido por duplicidade perceptual: {n_sidewalk_removidos}")
if len(sidewalk_disp) >= n_bg_alvo:
    idx = np.random.default_rng(SEED).choice(len(sidewalk_disp), size=n_bg_alvo, replace=False)
    sidewalk_imgs = [sidewalk_disp[i] for i in sorted(idx)]
else:
    sidewalk_imgs = sidewalk_disp
    print(f"AVISO: Sidewalk tem {len(sidewalk_disp)} imagens (< {n_bg_alvo}); background sera menor que 15%.")

# 4) Split agrupado 85/15 (positivos)
grupos = sorted(set(extrair_chave_grupo(nome_unico(img, pf)) for img, _, pf in pares_positivos))
grupos = list(np.random.default_rng(SEED).permutation(grupos))
grupos_train = set(grupos[: int(len(grupos) * 0.85)])

for img_path, lbl_path, prefixo in tqdm(pares_positivos, desc="Positivos"):
    nome_uniq = nome_unico(img_path, prefixo)
    g = extrair_chave_grupo(nome_uniq)
    treino = g in grupos_train
    shutil.copy(img_path, (UNIFIED_IMG_TRAIN if treino else UNIFIED_IMG_VAL) / nome_uniq)
    (UNIFIED_LBL_TRAIN if treino else UNIFIED_LBL_VAL).joinpath(
        Path(nome_uniq).stem + ".txt").write_text("\n".join(reescrever_label_lixo(lbl_path)))

# 5) Background com split proprio
bg_perm  = np.random.default_rng(SEED + 2).permutation(len(sidewalk_imgs))
corte_bg = int(len(sidewalk_imgs) * 0.85)
for split_idx, destino_img, destino_lbl in [
    (bg_perm[:corte_bg], UNIFIED_IMG_TRAIN, UNIFIED_LBL_TRAIN),
    (bg_perm[corte_bg:], UNIFIED_IMG_VAL,   UNIFIED_LBL_VAL),
]:
    for i in sorted(split_idx):
        img_path = sidewalk_imgs[i]
        shutil.copy(img_path, destino_img / f"sw_{img_path.name}")
        (destino_lbl / f"sw_{img_path.stem}.txt").touch()

n_train = contar_imagens(UNIFIED_IMG_TRAIN)
n_val   = contar_imagens(UNIFIED_IMG_VAL)
n_train_bg = sum(1 for f in UNIFIED_LBL_TRAIN.glob("*.txt") if f.stat().st_size == 0)
n_val_bg   = sum(1 for f in UNIFIED_LBL_VAL.glob("*.txt") if f.stat().st_size == 0)
print(f"Treino: {n_train} ({n_train_bg} bg, {n_train_bg/max(n_train,1):.1%}) | "
      f"Val: {n_val} ({n_val_bg} bg, {n_val_bg/max(n_val,1):.1%})")

In [ ]:
# data.yaml
DATA_YAML = Path("data/unified/data.yaml")
yaml.dump(
    {"path": str(Path("data/unified").resolve()),
     "train": "images/train", "val": "images/val",
     "nc": 1, "names": {0: "lixo"}},
    open(DATA_YAML, "w"), allow_unicode=True, sort_keys=False,
)
print(DATA_YAML.read_text())

## 4. Conjunto de teste

~560 imagens reais de ruas brasileiras (alvo 280 com residuos / 280 sem). O ground
truth binario e derivado da presenca de bounding boxes no `.txt` correspondente.
Em seguida, verificacao de vazamento treino/teste por pHash cruzado.

In [ ]:
TESTE_IMG, TESTE_LBL = Path("data/teste/images"), Path("data/teste/labels")
raw_test = Path("data/raw_imagens_teste")
for d in [TESTE_IMG, TESTE_LBL]:
    limpar_diretorio(d)

# Copiar imagens e labels (agregando todos os splits do export Roboflow)
for img_path in tqdm(listar_imagens(raw_test), desc="Copiando teste"):
    shutil.copy(img_path, TESTE_IMG / img_path.name)
    lbl_src = img_path.parent.parent / "labels" / (img_path.stem + ".txt")
    lbl_dst = TESTE_LBL / (img_path.stem + ".txt")
    (shutil.copy(lbl_src, lbl_dst) if lbl_src.exists() else lbl_dst.touch())

# Ground truth binario (presenca de bbox). Falha cedo se os labels nao foram copiados.
registros = []
for img_path in listar_imagens(TESTE_IMG):
    lbl = TESTE_LBL / (img_path.stem + ".txt")
    n_bboxes = len([l for l in lbl.read_text().strip().split("\n") if l.strip()]) if lbl.exists() else 0
    registros.append({"image_id": img_path.name, "tem_lixo": int(n_bboxes > 0), "n_bboxes": n_bboxes})
df_gt = pd.DataFrame(registros)
df_gt.to_csv("data/teste/ground_truth.csv", index=False)

n_com = int(df_gt["tem_lixo"].sum())
n_sem = len(df_gt) - n_com
assert n_com >= 100, f"Apenas {n_com} positivos no GT (esperado ~280). Verifique os labels do export."
print(f"Teste: {len(df_gt)} imagens | com residuos: {n_com} | sem residuos: {n_sem}")

In [ ]:
# Verificacao de vazamento treino/teste (pHash cruzado, distancia <= 5)
def hashes_de(paths):
    h = {}
    for p in tqdm(paths, desc="pHash cruzado", leave=False):
        try:
            h[imagehash.phash(Image.open(p).convert("RGB"))] = str(p)
        except Exception:
            continue
    return h

h_treino = hashes_de(listar_imagens(UNIFIED_IMG_TRAIN) + listar_imagens(UNIFIED_IMG_VAL))
h_teste  = hashes_de(listar_imagens(TESTE_IMG))
vazamentos = [(Path(t).name, ht - htr)
              for ht, t in h_teste.items() for htr in h_treino if ht - htr <= 5]
print(f"Vazamento treino/teste: {len(vazamentos)} possiveis "
      f"({'nenhum' if not vazamentos else 'INVESTIGAR'}).")

In [ ]:
# Exploracao do conjunto de teste
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(["Com residuos", "Sem residuos"], [n_com, n_sem], color=["#c0392b", "#27ae60"])
axes[0].set_title("Conjunto de teste"); axes[0].set_ylabel("Imagens")
for b in axes[0].patches:
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + 2, str(int(b.get_height())),
                 ha="center", fontsize=10)
pos = df_gt[df_gt["tem_lixo"] == 1]["n_bboxes"]
axes[1].hist(pos, bins=range(1, int(pos.max()) + 2), color="#c0392b", edgecolor="white")
axes[1].set_title("Bboxes por imagem positiva"); axes[1].set_xlabel("Numero de bounding boxes")
axes[1].set_ylabel("Frequencia")
plt.tight_layout(); plt.savefig("outputs/figures/fig_exploracao.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Treinamento YOLOv11m

Fine-tuning com classe unica `lixo` (`nc=1`) e ~15% de background. Treino
determinista (`seed=42`, `deterministic=True`). Se `best.pt` ja existir, e reutilizado.

In [ ]:
from ultralytics import YOLO

TRAIN_PROJECT = Path("runs/detect/models").resolve()
TRAIN_DIR     = TRAIN_PROJECT / "yolov11m_lixo"
BEST_PT       = TRAIN_DIR / "weights/best.pt"

if BEST_PT.exists():
    print(f"[skip] Reutilizando pesos existentes: {BEST_PT}")
    model_yolo = YOLO(str(BEST_PT))
else:
    model_yolo = YOLO(os.getenv("YOLO_PRETRAINED", "yolo11m.pt"))
    model_yolo.train(
        data=str(DATA_YAML.resolve()),
        epochs=100, batch=12, imgsz=640,
        optimizer="AdamW", lr0=0.001, patience=20,
        mosaic=1.0, fliplr=0.5, flipud=0.0,
        seed=SEED, deterministic=True,
        project=str(TRAIN_PROJECT), name="yolov11m_lixo",
        exist_ok=True, device=0, workers=4, verbose=True,
    )
    assert BEST_PT.exists(), f"best.pt nao encontrado apos o treino: {BEST_PT}"
    model_yolo = YOLO(str(BEST_PT))
print("Modelo YOLO pronto.")

In [ ]:
# Curvas de treinamento (se disponiveis)
RESULTS_CSV = TRAIN_DIR / "results.csv"
if RESULTS_CSV.exists():
    df_treino = pd.read_csv(RESULTS_CSV)
    df_treino.columns = df_treino.columns.str.strip()
    fig, axes = plt.subplots(2, 3, figsize=(16, 8)); axes = axes.flatten()
    plot_specs = [
        ("train/box_loss", "Loss Box (treino)",    "#c0392b"),
        ("train/cls_loss", "Loss Classe (treino)", "#e67e22"),
        ("val/box_loss",   "Loss Box (val)",       "#2980b9"),
        ("val/cls_loss",   "Loss Classe (val)",    "#1a5276"),
        ("metrics/precision(B)", "Precisao (val)",  "#7d3c98"),
    ]
    for ax, (col, titulo, cor) in zip(axes, plot_specs):
        if col in df_treino.columns:
            ax.plot(df_treino["epoch"], df_treino[col], color=cor, linewidth=1.5)
            ax.set_title(titulo); ax.set_xlabel("Epoca"); ax.grid(alpha=0.3)
    axes[-1].axis("off")
    plt.suptitle("Curvas de treinamento YOLOv11m (classe unica: lixo)", fontsize=11)
    plt.tight_layout(); plt.savefig("outputs/figures/fig_curvas_treino.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("results.csv nao encontrado (pesos pre-treinados reutilizados).")

## 6. Inferencia YOLO e latencia

**Metodologia de latencia (corrigida).** Duas definicoes, medidas de forma identica
para os dois modelos:

- **Inferencia** — tempo do modelo transformar uma imagem ja em memoria numa predicao.
  No YOLO usa-se `results.speed` do Ultralytics (preprocess + inference + postprocess),
  com `torch.cuda.synchronize()` em volta (a execucao em GPU e assincrona). No VLM e o
  round-trip da chamada ao servidor (inclui HTTP/serializacao, inerente ao LM Studio).
- **End-to-end** — pipeline completo a partir da leitura do arquivo em disco ate a
  predicao final. No YOLO inclui leitura/decodificacao robusta da imagem; no VLM inclui leitura, conversao e base64.

A inferencia roda com `conf=0.05` para capturar todas as deteccoes; a varredura de
limiares (0.25 / 0.35 / 0.50 / 0.65) filtra as deteccoes salvas sem reinferir. O limiar
principal e **0.25** (padrao da literatura YOLO), fixado a priori.

In [ ]:
test_images = listar_imagens(TESTE_IMG)
print(f"Imagens de teste: {len(test_images)}")
assert test_images, "Nenhuma imagem de teste encontrada. Execute a preparacao do conjunto de teste."

# Warm-up sincronizado (estabiliza kernels CUDA antes de medir)
_wup = ler_imagem_bgr(test_images[0])
for _ in range(5):
    model_yolo.predict(_wup, conf=0.05, iou=0.45, verbose=False)
torch.cuda.synchronize()

yolo_preds = []
for img_path in tqdm(test_images, desc="YOLO"):
    t_e2e0 = time.perf_counter()
    img = ler_imagem_bgr(img_path)                  # leitura + decode (entra no end-to-end)
    torch.cuda.synchronize()
    t_m0 = time.perf_counter()
    results = model_yolo.predict(img, conf=0.05, iou=0.45, verbose=False)
    torch.cuda.synchronize()                        # garante termino do kernel antes de medir
    t_model_manual = (time.perf_counter() - t_m0) * 1000
    t_e2e = (time.perf_counter() - t_e2e0) * 1000

    sp = results[0].speed                           # ms medidos pelo Ultralytics
    t_infer = float(sp.get("preprocess", 0) + sp.get("inference", 0) + sp.get("postprocess", 0))
    r = results[0]
    yolo_preds.append({
        "image_id": img_path.name,
        "deteccoes": [{"confidence": float(b.conf), "bbox_xywhn": b.xywhn[0].tolist()} for b in r.boxes],
        "latencia_inferencia_ms":   round(t_infer, 2),         # preprocess+inference+postprocess
        "latencia_model_manual_ms": round(t_model_manual, 2),  # wall-clock sincronizado (cross-check)
        "latencia_end_to_end_ms":   round(t_e2e, 2),           # inclui leitura/decode
    })

json.dump({"hardware": GPU_NAME, "predicoes": yolo_preds},
          open("outputs/predictions_yolo.json", "w"), indent=2)

lat_y_infer = [p["latencia_inferencia_ms"] for p in yolo_preds]
lat_y_e2e   = [p["latencia_end_to_end_ms"] for p in yolo_preds]
print(f"YOLO inferencia : media={np.mean(lat_y_infer):.1f} P50={np.percentile(lat_y_infer,50):.1f} "
      f"P95={np.percentile(lat_y_infer,95):.1f} ms ({1000/np.mean(lat_y_infer):.1f} img/s)")
print(f"YOLO end-to-end : media={np.mean(lat_y_e2e):.1f} P50={np.percentile(lat_y_e2e,50):.1f} "
      f"P95={np.percentile(lat_y_e2e,95):.1f} ms")

In [ ]:
# Varredura de limiares (analise de sensibilidade — nao e criterio de selecao)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

def imagem_tem_lixo(deteccoes, limiar):
    return int(any(d["confidence"] >= limiar for d in deteccoes))

gt_map     = dict(zip(df_gt["image_id"], df_gt["tem_lixo"]))
ids_ok     = [p["image_id"] for p in yolo_preds if p["image_id"] in gt_map]
y_true     = [gt_map[i] for i in ids_ok]
pred_by_id = {p["image_id"]: p for p in yolo_preds}

LIMIARES = [0.25, 0.35, 0.50, 0.65]
linhas = []
for limiar in LIMIARES:
    y_pred = [imagem_tem_lixo(pred_by_id[i]["deteccoes"], limiar) for i in ids_ok]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    linhas.append({
        "limiar": limiar,
        "acuracia":       round(accuracy_score(y_true, y_pred), 4),
        "precisao":       round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":         round(recall_score(y_true, y_pred, zero_division=0), 4),
        "especificidade": round(tn / (tn + fp), 4) if (tn + fp) else 0,
        "f1":             round(f1_score(y_true, y_pred, zero_division=0), 4),
        "taxa_fp":        round(fp / (fp + tn), 4) if (fp + tn) else 0,
        "fp": int(fp), "fn": int(fn), "tp": int(tp), "tn": int(tn),
    })
df_limiar = pd.DataFrame(linhas)
df_limiar.to_csv("outputs/varredura_limiares.csv", index=False)
print(df_limiar.to_string(index=False))

In [ ]:
# Figura: sensibilidade ao limiar
fig, ax = plt.subplots(figsize=(9, 5))
for metrica, cor in {"f1": "#1e8449", "recall": "#2980b9",
                     "especificidade": "#7d3c98", "taxa_fp": "#c0392b"}.items():
    ax.plot(df_limiar["limiar"], df_limiar[metrica], marker="o", label=metrica, color=cor, linewidth=2)
ax.axvline(x=0.25, color="gray", linestyle="--", linewidth=1.5, label="limiar principal (0.25)")
ax.set_xlabel("Limiar de confianca"); ax.set_ylabel("Valor")
ax.set_title("Sensibilidade ao limiar de deteccao (YOLOv11m)")
ax.set_xticks(LIMIARES); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("outputs/figures/fig_varredura_limiares.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Inferencia Gemma 3 27B (VLM zero-shot)

Via LM Studio (endpoint compativel com a API OpenAI). `temperature=0`; `seed=42` e
`response_format: json_schema` sao usados quando o servidor os suporta. O **prompt e
versionado** (hash) e respostas invalidas sao contabilizadas. Cache por imagem evita
reprocessar a inferencia (custosa) entre execucoes.

In [ ]:
import requests
from openai import OpenAI

LM_BASE_URL = os.getenv("LM_STUDIO_BASE_URL", "http://localhost:1234/v1")
r = requests.get(f"{LM_BASE_URL}/models", timeout=5); r.raise_for_status()
disponiveis = [m["id"] for m in r.json().get("data", [])]

modelo_esperado = os.getenv("LM_STUDIO_MODEL")
modelo_vlm = next((m for m in disponiveis if m == modelo_esperado or m.startswith(modelo_esperado)), None)
if modelo_vlm is None:
    raise RuntimeError(f"Modelo '{modelo_esperado}' nao carregado no LM Studio. Disponiveis: {disponiveis}")
client = OpenAI(base_url=LM_BASE_URL, api_key="lm-studio")
print(f"Modelo VLM: {modelo_vlm}")

# Deteccao de suporte a seed e a json_schema (um teste rapido de cada)
def _suporta(**extra):
    try:
        client.chat.completions.create(
            model=modelo_vlm, messages=[{"role": "user", "content": "ping"}],
            max_tokens=1, temperature=0, **extra)
        return True
    except Exception:
        return False

VLM_SUPORTA_SEED = _suporta(seed=SEED)
VLM_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {"name": "lixo_response", "schema": {
        "type": "object", "required": ["tem_lixo"],
        "properties": {"tem_lixo": {"type": "boolean"},
                       "descricao_breve": {"type": "string"},
                       "confianca": {"type": "number"}}}},
}
if not _suporta(response_format=VLM_RESPONSE_FORMAT):
    VLM_RESPONSE_FORMAT = None
print(f"Suporte do servidor: seed={VLM_SUPORTA_SEED}, json_schema={VLM_RESPONSE_FORMAT is not None}")

In [ ]:
# Prompt versionado (o texto exato e parte da reprodutibilidade do resultado zero-shot)
import jsonschema

PROMPT_VLM = """You are a public street monitoring analyst.
Your task is to answer one single question: IS THERE TRASH or discarded WASTE in this street image?

Consider as trash: garbage bags, bottles, cans, cardboard, packaging, plastics,
glass, or any discarded waste on a public street or sidewalk.

Do NOT consider as trash: closed trash bins/containers in normal use, vehicles, people,
vegetation, street furniture in good condition, or normal pavement.

Respond STRICTLY in valid JSON, with no markdown and no text outside the JSON.
Use exactly these field names:
{"tem_lixo": true, "descricao_breve": "...", "confianca": 0.0}

If there is no visible trash, return tem_lixo: false."""

SCHEMA_VLM = {
    "type": "object", "required": ["tem_lixo"],
    "properties": {"tem_lixo": {"type": "boolean"},
                   "descricao_breve": {"type": "string"},
                   "confianca": {"type": "number"}},
}
PROMPT_HASH = hashlib.sha256(PROMPT_VLM.encode()).hexdigest()[:16]
VLM_CACHE_DIR = Path("outputs/vlm_cache")
print(f"Prompt hash: {PROMPT_HASH}")

In [ ]:
def parse_json_robusto(raw: str) -> dict:
    if not raw:
        return {"parse_error": True}
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        i, j = raw.find("{"), raw.rfind("}") + 1
        if 0 <= i < j:
            try:
                return json.loads(raw[i:j])
            except json.JSONDecodeError:
                pass
    return {"parse_error": True}

def metadados_vlm_cache(image_path):
    image_path = Path(image_path)
    return {
        "prompt_hash": PROMPT_HASH,
        "modelo_vlm": modelo_vlm,
        "lm_studio_base_url": LM_BASE_URL,
        "seed": SEED if VLM_SUPORTA_SEED else None,
        "temperature": 0,
        "max_tokens": 300,
        "response_format": "json_schema" if VLM_RESPONSE_FORMAT is not None else None,
        "image_id": image_path.name,
        "image_sha256": sha256_arquivo(image_path),
    }

def cache_path_vlm(image_path):
    image_path = Path(image_path)
    chave_nome = hashlib.sha256(image_path.name.encode("utf-8")).hexdigest()[:12]
    return VLM_CACHE_DIR / f"{image_path.stem}_{chave_nome}.json"

def inferir_vlm(image_path: str, max_tentativas: int = 3) -> dict:
    # End-to-end = preparo (leitura + base64) + inferencia (round-trip HTTP).
    image_path = Path(image_path)
    t_prep0 = time.perf_counter()
    imagem_payload = imagem_para_data_url_jpeg(image_path)
    cache_meta = metadados_vlm_cache(image_path)
    t_prep = (time.perf_counter() - t_prep0) * 1000

    kwargs = dict(
        model=modelo_vlm,
        messages=[{"role": "user", "content": [
            {"type": "text",      "text": PROMPT_VLM},
            {"type": "image_url", "image_url": {"url": imagem_payload["data_url"]}},
        ]}],
        max_tokens=300, temperature=0, timeout=45,
    )
    if VLM_SUPORTA_SEED:
        kwargs["seed"] = SEED
    if VLM_RESPONSE_FORMAT is not None:
        kwargs["response_format"] = VLM_RESPONSE_FORMAT

    ultimo_erro = None
    for tentativa in range(max_tentativas):
        try:
            t0 = time.perf_counter()
            resp = client.chat.completions.create(**kwargs)
            t_infer = (time.perf_counter() - t0) * 1000
            raw = resp.choices[0].message.content
            parsed = parse_json_robusto(raw)
            try:
                jsonschema.validate(parsed, SCHEMA_VLM); schema_ok = True
            except jsonschema.ValidationError:
                schema_ok = False
            return {
                "tem_lixo":        bool(parsed.get("tem_lixo", False)),
                "descricao_breve": parsed.get("descricao_breve", ""),
                "confianca":       parsed.get("confianca", 0.0),
                "parse_error":     bool(parsed.get("parse_error", False)),
                "schema_valido":   schema_ok,
                "raw_response":    raw,
                **cache_meta,
                "image_mime_type": imagem_payload["mime_type"],
                "image_payload_bytes": imagem_payload["payload_bytes"],
                "image_payload_sha256": imagem_payload["payload_sha256"],
                "latencia_inferencia_ms": round(t_infer, 2),
                "latencia_end_to_end_ms": round(t_prep + t_infer, 2),
            }
        except Exception as e:
            ultimo_erro = str(e)
            time.sleep(2 ** tentativa)
    return {"tem_lixo": False, "parse_error": True, "erro": ultimo_erro,
            **metadados_vlm_cache(image_path),
            "latencia_inferencia_ms": 0.0, "latencia_end_to_end_ms": 0.0}

In [ ]:
# Execucao do VLM (cache por imagem + modelo + prompt + parametros + hash do arquivo)
def _cache_valido(p: Path, img_path: Path) -> bool:
    try:
        salvo = json.loads(p.read_text())
        esperado = metadados_vlm_cache(img_path)
        return all(salvo.get(k) == v for k, v in esperado.items())
    except Exception:
        return False

pendentes = [img for img in test_images if not _cache_valido(cache_path_vlm(img), img)]
print(f"VLM: {len(pendentes)} pendentes de {len(test_images)}.")

if pendentes:
    inferir_vlm(str(pendentes[0]))  # warm-up
for img_path in tqdm(pendentes, desc="VLM"):
    res = inferir_vlm(str(img_path))
    res["image_id"] = img_path.name
    json.dump(res, open(cache_path_vlm(img_path), "w"), ensure_ascii=False, indent=2)

vlm_preds = [json.loads(cache_path_vlm(img).read_text())
             for img in test_images if cache_path_vlm(img).exists()]
assert len(vlm_preds) == len(test_images), f"Predicoes VLM incompletas: {len(vlm_preds)}/{len(test_images)}"
json.dump(vlm_preds, open("outputs/predictions_vlm.json", "w"), ensure_ascii=False, indent=2)

lat_v_infer = [p["latencia_inferencia_ms"] for p in vlm_preds if not p.get("erro")]
lat_v_e2e   = [p["latencia_end_to_end_ms"] for p in vlm_preds if not p.get("erro")]
n_erros_vlm = sum(1 for p in vlm_preds if p.get("parse_error") or p.get("erro"))
print(f"VLM inferencia : media={np.mean(lat_v_infer):.1f} P50={np.percentile(lat_v_infer,50):.1f} "
      f"P95={np.percentile(lat_v_infer,95):.1f} ms ({1000/np.mean(lat_v_infer):.3f} img/s)")
print(f"VLM end-to-end : media={np.mean(lat_v_e2e):.1f} ms | respostas invalidas: {n_erros_vlm}/{len(vlm_preds)}")

## 8. Avaliacao comparativa

Decisao binaria por imagem (limiar YOLO = 0.25). Metricas, matrizes de confusao,
trade-off acuracia x latencia, teste de McNemar pareado, ICs de Wilson e MCC/kappa.

In [ ]:
def metricas_binarias(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "acuracia":       round(accuracy_score(y_true, y_pred), 4),
        "precisao":       round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":         round(recall_score(y_true, y_pred, zero_division=0), 4),
        "especificidade": round(tn / (tn + fp), 4) if (tn + fp) else 0,
        "f1":             round(f1_score(y_true, y_pred, zero_division=0), 4),
        "taxa_fp":        round(fp / (fp + tn), 4) if (fp + tn) else 0,
        "matriz_confusao": [[int(tn), int(fp)], [int(fn), int(tp)]],
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    }

def vlm_decisao(p):
    if p.get("parse_error") or p.get("erro"):
        return 0
    return int(bool(p.get("tem_lixo")))

LIMIAR_ESCOLHIDO = 0.25
vlm_by_id = {p["image_id"]: p for p in vlm_preds}
ids       = [i for i in ids_ok if i in vlm_by_id]
y_true_c  = [gt_map[i] for i in ids]
y_pred_v  = [vlm_decisao(vlm_by_id[i]) for i in ids]
y_pred_y  = [imagem_tem_lixo(pred_by_id[i]["deteccoes"], LIMIAR_ESCOLHIDO) for i in ids]

met_yolo = metricas_binarias(y_true_c, y_pred_y)
met_vlm  = metricas_binarias(y_true_c, y_pred_v)

print(f"{'Metrica':<16}{'YOLOv11m':>11}{'Gemma 3 27B':>14}")
print("-" * 41)
for k in ["acuracia", "precisao", "recall", "especificidade", "f1", "taxa_fp"]:
    print(f"  {k:<14}{met_yolo[k]:>11.4f}{met_vlm[k]:>14.4f}")
print(f"\nRespostas invalidas do VLM (contadas como 'sem lixo'): {n_erros_vlm}")

In [ ]:
# Matrizes de confusao
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, met, titulo, cmap in [
    (axes[0], met_yolo, f"YOLOv11m (limiar={LIMIAR_ESCOLHIDO})", "Blues"),
    (axes[1], met_vlm,  "Gemma 3 27B (zero-shot)", "Greens"),
]:
    sns.heatmap(np.array(met["matriz_confusao"]), annot=True, fmt="d", cmap=cmap, ax=ax, cbar=False,
                xticklabels=["Prev: Sem", "Prev: Com"], yticklabels=["Real: Sem", "Real: Com"])
    ax.set_title(titulo)
plt.suptitle("Matrizes de confusao — deteccao binaria", fontsize=12)
plt.tight_layout(); plt.savefig("outputs/figures/fig_matrizes_confusao.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Barras: metricas binarias
metricas_b = ["acuracia", "precisao", "recall", "especificidade", "f1"]
x, w = np.arange(len(metricas_b)), 0.35
fig, ax = plt.subplots(figsize=(11, 5))
bars_y = ax.bar(x - w/2, [met_yolo[m] for m in metricas_b], w, label="YOLOv11m",    color="#2980b9")
bars_v = ax.bar(x + w/2, [met_vlm[m]  for m in metricas_b], w, label="Gemma 3 27B", color="#1e8449")
for b in list(bars_y) + list(bars_v):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005, f"{b.get_height():.3f}",
            ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metricas_b, rotation=10); ax.set_ylim(0, 1.1)
ax.set_ylabel("Valor"); ax.set_title("YOLOv11m vs Gemma 3 27B — metricas binarias")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("outputs/figures/fig_metricas_comparativo.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Latencia: boxplot end-to-end (mesma definicao para ambos)
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([lat_y_e2e, lat_v_e2e],
           labels=["YOLOv11m (end-to-end)", "Gemma 3 27B (end-to-end)"],
           patch_artist=True, boxprops=dict(facecolor="#aed6f1", alpha=0.7),
           medianprops=dict(color="black", linewidth=2))
ax.set_ylabel("Latencia (ms)"); ax.set_yscale("log")
ax.set_title("Distribuicao de latencia por imagem — end-to-end (escala log)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("outputs/figures/fig_latencia_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

fator_e2e   = np.mean(lat_v_e2e)   / np.mean(lat_y_e2e)
fator_infer = np.mean(lat_v_infer) / np.mean(lat_y_infer)
print(f"YOLO mais rapido que o VLM: {fator_e2e:.0f}x (end-to-end) | {fator_infer:.0f}x (inferencia).")

In [ ]:
# Figura de trade-off: F1 x latencia end-to-end (eixo log). "Melhor" = canto superior esquerdo.
fig, ax = plt.subplots(figsize=(8, 5.5))
pontos = [
    ("YOLOv11m (limiar=0.25)", np.mean(lat_y_e2e), met_yolo["f1"], "#2980b9"),
    ("Gemma 3 27B (zero-shot)", np.mean(lat_v_e2e), met_vlm["f1"], "#1e8449"),
]
for nome, lat, f1, cor in pontos:
    ax.scatter(lat, f1, s=180, color=cor, edgecolor="black", zorder=3, label=nome)
    ax.annotate(f"{nome}\nF1={f1:.3f} | {lat:.0f} ms",
                (lat, f1), textcoords="offset points", xytext=(10, -6), fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("Latencia end-to-end por imagem (ms, escala log)")
ax.set_ylabel("F1-score")
ax.set_title("Trade-off acuracia x custo computacional")
ax.grid(alpha=0.3, which="both")
ax.annotate("melhor", xy=(0.04, 0.96), xycoords="axes fraction",
            ha="left", va="top", fontsize=9, color="gray", style="italic")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout(); plt.savefig("outputs/figures/fig_tradeoff_f1_latencia.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Tabela comparativa final
CHAVES = ["acuracia", "precisao", "recall", "especificidade", "f1", "taxa_fp"]
df_comparativo = pd.DataFrame({
    "metrica":      CHAVES,
    "YOLOv11m":    [met_yolo[k] for k in CHAVES],
    "Gemma_3_27B": [met_vlm[k]  for k in CHAVES],
})
df_comparativo.to_csv("outputs/tabela_comparativa_final.csv", index=False)
print(df_comparativo.to_string(index=False))

In [ ]:
# Significancia e intervalos de confianca
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef, balanced_accuracy_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test

# McNemar pareado (discordancias entre os acertos dos dois modelos)
ac_y = np.array([int(p == t) for p, t in zip(y_pred_y, y_true_c)])
ac_v = np.array([int(p == t) for p, t in zip(y_pred_v, y_true_c)])
n11 = int(np.sum((ac_y == 1) & (ac_v == 1)))
n10 = int(np.sum((ac_y == 1) & (ac_v == 0)))
n01 = int(np.sum((ac_y == 0) & (ac_v == 1)))
n00 = int(np.sum((ac_y == 0) & (ac_v == 0)))
res_mcnemar = mcnemar_test([[n11, n10], [n01, n00]], exact=False, correction=True)
print(f"McNemar: chi2={res_mcnemar.statistic:.2f}, p={res_mcnemar.pvalue:.3e} "
      f"(so YOLO acerta={n10}, so Gemma acerta={n01})")

# ICs de Wilson 95%
n_total = len(y_true_c)
def ic_wilson(sucessos, total, nome):
    if total == 0:
        return
    low, high = proportion_confint(sucessos, total, alpha=0.05, method="wilson")
    print(f"  {nome:<14} {sucessos/total:.4f} [{low:.4f} - {high:.4f}]")

for nome, met in [("YOLOv11m", met_yolo), ("Gemma 3 27B", met_vlm)]:
    print(f"\nIC 95% Wilson — {nome}:")
    ic_wilson(met["tp"] + met["tn"], n_total,            "acuracia")
    ic_wilson(met["tp"], met["tp"] + met["fn"],          "recall")
    ic_wilson(met["tn"], met["tn"] + met["fp"],          "especificidade")
    ic_wilson(met["tp"], met["tp"] + met["fp"],          "precisao")
    ic_wilson(met["fp"], met["fp"] + met["tn"],          "taxa_fp")

# MCC / kappa / acuracia balanceada
print()
estat = {"mcnemar": {"chi2": float(res_mcnemar.statistic), "p_value": float(res_mcnemar.pvalue),
                     "n11": n11, "n10": n10, "n01": n01, "n00": n00}}
for nome, chave, y_pred in [("YOLOv11m", "yolo", y_pred_y), ("Gemma 3 27B", "gemma", y_pred_v)]:
    kappa = cohen_kappa_score(y_true_c, y_pred)
    mcc   = matthews_corrcoef(y_true_c, y_pred)
    bal   = balanced_accuracy_score(y_true_c, y_pred)
    print(f"{nome}: kappa={kappa:.3f}, MCC={mcc:.3f}, acc_balanceada={bal:.3f}, Youden_J={2*bal-1:.3f}")
    estat[chave] = {"kappa": float(kappa), "mcc": float(mcc), "acc_balanceada": float(bal)}

json.dump(estat, open("outputs/estatisticas_inferenciais.json", "w"), indent=2, ensure_ascii=False)
print("\nEstatisticas salvas em outputs/estatisticas_inferenciais.json")

## 9. Analise qualitativa

Classificacao de cada imagem por tipo de concordancia/erro entre os modelos e figura
consolidada de exemplos (insumo para a discussao de erros do paper).

In [ ]:
def classificar_caso(gt, py, pv):
    if gt == 1 and py == 1 and pv == 1: return "ambos_tp"
    if gt == 0 and py == 0 and pv == 0: return "ambos_tn"
    if gt == 1 and py == 1 and pv == 0: return "yolo_ok_vlm_fn"
    if gt == 1 and py == 0 and pv == 1: return "vlm_ok_yolo_fn"
    if gt == 0 and py == 1 and pv == 0: return "yolo_fp"
    if gt == 0 and py == 0 and pv == 1: return "vlm_fp"
    if gt == 0 and py == 1 and pv == 1: return "ambos_fp"
    if gt == 1 and py == 0 and pv == 0: return "ambos_fn"
    return "outro"

df_casos = pd.DataFrame([{
    "image_id": i, "gt": gt_map[i],
    "pred_yolo": imagem_tem_lixo(pred_by_id[i]["deteccoes"], LIMIAR_ESCOLHIDO),
    "pred_vlm":  vlm_decisao(vlm_by_id[i]),
    "caso": classificar_caso(gt_map[i],
                             imagem_tem_lixo(pred_by_id[i]["deteccoes"], LIMIAR_ESCOLHIDO),
                             vlm_decisao(vlm_by_id[i])),
} for i in ids])
df_casos.to_csv("outputs/casos_qualitativos.csv", index=False)
print(df_casos["caso"].value_counts().to_string())

In [ ]:
# Figura consolidada: um exemplo por tipo de caso
tipos = [("ambos_tp", "TP ambos"), ("ambos_tn", "TN ambos"),
         ("yolo_ok_vlm_fn", "YOLO ok / VLM FN"), ("vlm_ok_yolo_fn", "VLM ok / YOLO FN"),
         ("ambos_fp", "FP ambos"), ("ambos_fn", "FN ambos"),
         ("yolo_fp", "FP so YOLO"), ("vlm_fp", "FP so VLM")]
fig, axes = plt.subplots(2, 4, figsize=(18, 9)); axes = axes.flatten()
for ax, (tipo, leg) in zip(axes, tipos):
    sub = df_casos[df_casos["caso"] == tipo]
    if sub.empty:
        ax.text(0.5, 0.5, "N/A", ha="center", va="center"); ax.set_title(leg); ax.axis("off"); continue
    row = sub.iloc[0]
    ax.imshow(np.array(Image.open(TESTE_IMG / row["image_id"]).convert("RGB"))); ax.axis("off")
    ax.set_title(f"{leg} | Y={row['pred_yolo']} V={row['pred_vlm']} GT={row['gt']}", fontsize=9)
plt.suptitle("Analise qualitativa — YOLOv11m vs Gemma 3 27B", fontsize=12)
plt.tight_layout(); plt.savefig("outputs/figures/fig_exemplos_qualitativos.png", dpi=300, bbox_inches="tight")
plt.show()

fp_yolo = int(df_casos["caso"].isin(["yolo_fp", "ambos_fp"]).sum())
fp_vlm  = int(df_casos["caso"].isin(["vlm_fp", "ambos_fp"]).sum())
fn_yolo = int(df_casos["caso"].isin(["ambos_fn", "vlm_ok_yolo_fn"]).sum())
fn_vlm  = int(df_casos["caso"].isin(["ambos_fn", "yolo_ok_vlm_fn"]).sum())
print(f"FP: YOLO={fp_yolo}, VLM={fp_vlm} | FN: YOLO={fn_yolo}, VLM={fn_vlm}")

## 10. Exportacao dos resultados

Consolida metricas, latencias (ambas as definicoes), significancia e metadados de
reprodutibilidade em `outputs/metricas_finais.json`.

In [ ]:
def _rf_config(prefixo):
    return {
        "workspace": os.getenv(f"ROBOFLOW_WORKSPACE_{prefixo}"),
        "project": os.getenv(f"ROBOFLOW_PROJECT_{prefixo}"),
        "version": os.getenv(f"ROBOFLOW_VERSION_{prefixo}"),
    }

manifestos_dados = {
    "treino_validacao_unificado": hash_manifesto_diretorio("data/unified", EXTS_MANIFESTO_DADOS),
    "teste_final": hash_manifesto_diretorio("data/teste", EXTS_MANIFESTO_DADOS),
}

metadados_reprodutibilidade = {
    "python": sys.version,
    "platform": sys.platform,
    "pacotes": versoes_pacotes(),
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "determinismo": {
        "PYTHONHASHSEED": os.getenv("PYTHONHASHSEED"),
        "seed": SEED,
        "torch_cudnn_deterministic": torch.backends.cudnn.deterministic,
        "torch_cudnn_benchmark": torch.backends.cudnn.benchmark,
    },
    "roboflow": {
        "trash": _rf_config("TRASH"),
        "sidewalk": _rf_config("SIDEWALK"),
        "teste": _rf_config("TESTE"),
    },
    "taco": {
        "repo_url": "https://github.com/pedropro/TACO.git",
        "commit": git_commit(Path("data/raw_taco/TACO")),
    },
    "dados": manifestos_dados,
    "checkpoint_yolo_best_pt_sha256": sha256_arquivo(BEST_PT) if BEST_PT.exists() else None,
}

resumo = {
    "projeto": "YOLOv11m vs Gemma 3 27B — deteccao binaria de residuos em vias publicas",
    "data_execucao": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "hardware": GPU_NAME,
    "seed": SEED,
    "reprodutibilidade": metadados_reprodutibilidade,
    "treino": {
        "datasets_positivos": ["TACO (COCO->YOLO)", "Trash Detection (Roboflow)"],
        "background": "Sidewalk Segmentation",
        "n_train_total": n_train, "n_train_bg": n_train_bg,
        "proporcao_background": f"{n_train_bg/max(n_train,1):.1%}",
        "hiperparametros": {"modelo": os.getenv("YOLO_PRETRAINED", "yolo11m.pt"),
                            "epochs": 100, "batch": 12, "imgsz": 640,
                            "optimizer": "AdamW", "lr0": 0.001, "patience": 20,
                            "seed": SEED, "deterministic": True},
    },
    "teste": {"fonte": "~560 imagens reais de ruas brasileiras",
              "total": len(df_gt), "com_lixo": n_com, "sem_lixo": n_sem,
              "anotacao": "por regiao de descarte; GT binario pela presenca de bbox"},
    "yolo": {"limiar_principal": LIMIAR_ESCOLHIDO, **met_yolo,
             "latencia_inferencia_media_ms": round(np.mean(lat_y_infer), 2),
             "latencia_inferencia_p95_ms":   round(np.percentile(lat_y_infer, 95), 2),
             "latencia_end_to_end_media_ms": round(np.mean(lat_y_e2e), 2),
             "throughput_inferencia_img_s":  round(1000 / np.mean(lat_y_infer), 1)},
    "vlm": {"modelo": modelo_vlm, "prompt_hash": PROMPT_HASH,
            "seed_suportada": VLM_SUPORTA_SEED, "json_schema_suportado": VLM_RESPONSE_FORMAT is not None,
            "respostas_invalidas": n_erros_vlm, **met_vlm,
            "latencia_inferencia_media_ms": round(np.mean(lat_v_infer), 2),
            "latencia_inferencia_p95_ms":   round(np.percentile(lat_v_infer, 95), 2),
            "latencia_end_to_end_media_ms": round(np.mean(lat_v_e2e), 2),
            "throughput_inferencia_img_s":  round(1000 / np.mean(lat_v_infer), 4)},
    "comparativo": {
        "fator_velocidade_end_to_end": round(np.mean(lat_v_e2e)   / np.mean(lat_y_e2e), 1),
        "fator_velocidade_inferencia": round(np.mean(lat_v_infer) / np.mean(lat_y_infer), 1),
        "nota_latencia": "VLM via LM Studio (HTTP); YOLO em processo. Comparacao de estrategias de implantacao.",
        "diferenca_f1":      round(met_vlm["f1"]     - met_yolo["f1"], 4),
        "diferenca_taxa_fp": round(met_yolo["taxa_fp"] - met_vlm["taxa_fp"], 4),
        "mcnemar_chi2": float(res_mcnemar.statistic), "mcnemar_p": float(res_mcnemar.pvalue),
    },
}
json.dump(resumo, open("outputs/metricas_finais.json", "w"), ensure_ascii=False, indent=2)

print("=" * 56)
print("RESUMO FINAL")
print("=" * 56)
print(f"Hardware: {GPU_NAME} | seed={SEED}")
print(f"Teste: {len(df_gt)} ({n_com} com / {n_sem} sem residuos)")
print(f"\n{'Metrica':<16}{'YOLOv11m':>11}{'Gemma 3 27B':>14}")
print("-" * 41)
for k in ["acuracia", "precisao", "recall", "especificidade", "f1", "taxa_fp"]:
    print(f"  {k:<14}{met_yolo[k]:>11.4f}{met_vlm[k]:>14.4f}")
print(f"\nLatencia inferencia: YOLO {np.mean(lat_y_infer):.1f} ms | VLM {np.mean(lat_v_infer):.1f} ms")
print(f"Latencia end-to-end: YOLO {np.mean(lat_y_e2e):.1f} ms | VLM {np.mean(lat_v_e2e):.1f} ms")
print(f"Fator de velocidade: {resumo['comparativo']['fator_velocidade_end_to_end']:.0f}x (end-to-end)")
print(f"McNemar: chi2={res_mcnemar.statistic:.2f}, p={res_mcnemar.pvalue:.2e}")
print("=" * 56)

---
**Execucao completa.** Artefatos em `outputs/`.

**Proximos passos (notebook 1, opcionais — ver plano estrategico):**
baseline CLIP zero-shot nas mesmas 560 imagens (terceiro ponto no trade-off) e analise
de taxa de sucesso de localizacao do VLM (IoU > 0.5 sobre os verdadeiros positivos).